<a href="https://colab.research.google.com/github/Mandeep8252/AI-LAB/blob/main/AI_Week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import heapq

start = [2, 8, 3,
         1, 6, 4,
         7, 0, 5]

goal = [1, 2, 3,
        8, 0, 4,
        7, 6, 5]


def manhattan_distance(state, goal):
    """Calculates the sum of Manhattan distances of tiles from their goal positions (excluding the blank space 0)."""
    distance = 0
    for i in range(9):
        tile = state[i]
        if tile != 0:
            # Current 2D coordinates
            curr_row, curr_col = i // 3, i % 3

            # Goal 2D coordinates
            goal_index = goal.index(tile)
            goal_row, goal_col = goal_index // 3, goal_index % 3

            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
    return distance


def misplaced_tiles(state, goal):
    """Calculates the number of misplaced tiles (excluding the blank space 0)."""
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != goal[i]:
            count += 1
    return count


def get_neighbors(state):
    neighbors = []

    blank = state.index(0)
    row = blank // 3
    col = blank % 3

    if row > 0:
        new_state = state.copy()
        new_state[blank], new_state[blank - 3] = \
            new_state[blank - 3], new_state[blank]
        neighbors.append(new_state)

    if row < 2:
        new_state = state.copy()
        new_state[blank], new_state[blank + 3] = \
            new_state[blank + 3], new_state[blank]
        neighbors.append(new_state)

    if col > 0:
        new_state = state.copy()
        new_state[blank], new_state[blank - 1] = \
            new_state[blank - 1], new_state[blank]
        neighbors.append(new_state)

    if col < 2:
        new_state = state.copy()
        new_state[blank], new_state[blank + 1] = \
            new_state[blank + 1], new_state[blank]
        neighbors.append(new_state)

    return neighbors


def a_star(start, goal, heuristic_fn=misplaced_tiles):
    h = heuristic_fn(start, goal)
    initial_path = [(start, 0, h, h)]
    priority_queue = [(h, 0, start, initial_path)]
    visited = set()

    total_branches = 0
    states_checked = 0

    while priority_queue:
        f, g, current, path = heapq.heappop(priority_queue)
        current_tuple = tuple(current)

        if current_tuple in visited:
            continue

        visited.add(current_tuple)
        neighbors = get_neighbors(current)
        total_branches += len(neighbors)
        states_checked += 1

        if current == goal:
            branching_factor = total_branches / states_checked if states_checked > 0 else 0
            return path, len(visited), branching_factor

        for next_state in neighbors:
            if tuple(next_state) not in visited:
                new_g = g + 1
                new_h = heuristic_fn(next_state, goal)
                new_f = new_g + new_h

                next_step_info = (next_state, new_g, new_h, new_f)
                heapq.heappush(
                    priority_queue,
                    (new_f, new_g, next_state, path + [next_step_info])
                )

    return None, len(visited), 0


def print_puzzle(state):
    for i in range(0, 9, 3):
        print(state[i], state[i + 1], state[i + 2])
    print()


# Solve using Misplaced Tiles Heuristic
solution_path, explored_states, b = a_star(start, goal, heuristic_fn=misplaced_tiles)

if solution_path:
    print("Initial State:")
    print_puzzle(start)

    print("Goal State:")
    print_puzzle(goal)

    print("Solution:")
    print("Number of moves:", len(solution_path) - 1)
    print("Number of states:", len(solution_path))
    print("States explored:", explored_states)
    print()

    for i, (state, g, h, f) in enumerate(solution_path):
        print(f"Step {i} (Path Cost g={g}, Heuristic h={h}, Total f={f})")
        print_puzzle(state)

    d = len(solution_path) - 1
    time_complexity = b ** d
    space_complexity = b ** d

    print("Complexity:")
    print("Branching factor (b):", round(b, 2))
    print("Depth (d):", d)
    print(f"Time Complexity (b^d): {round(b, 2)} ^ {d} = {round(time_complexity, 2)}")
    print(f"Space Complexity (b^d): {round(b, 2)} ^ {d} = {round(space_complexity, 2)}")
else:
    print("No solution found.")

Initial State:
2 8 3
1 6 4
7 0 5

Goal State:
1 2 3
8 0 4
7 6 5

Solution:
Number of moves: 5
Number of states: 6
States explored: 7

Step 0 (Path Cost g=0, Heuristic h=4, Total f=4)
2 8 3
1 6 4
7 0 5

Step 1 (Path Cost g=1, Heuristic h=3, Total f=4)
2 8 3
1 0 4
7 6 5

Step 2 (Path Cost g=2, Heuristic h=3, Total f=5)
2 0 3
1 8 4
7 6 5

Step 3 (Path Cost g=3, Heuristic h=2, Total f=5)
0 2 3
1 8 4
7 6 5

Step 4 (Path Cost g=4, Heuristic h=1, Total f=5)
1 2 3
0 8 4
7 6 5

Step 5 (Path Cost g=5, Heuristic h=0, Total f=5)
1 2 3
8 0 4
7 6 5

Complexity:
Branching factor (b): 3.14
Depth (d): 5
Time Complexity (b^d): 3.14 ^ 5 = 306.64
Space Complexity (b^d): 3.14 ^ 5 = 306.64


In [6]:
import random

# Calculate number of attacking pairs (conflicts to minimize)
def heuristic(state):
    conflicts = 0
    for i in range(4):
        for j in range(i + 1, 4):
            # Same row
            if state[i] == state[j]:
                conflicts += 1
            # Same diagonal
            if abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts


# Generate all possible neighboring states
def get_neighbors(state):
    neighbors = []
    for col in range(4):
        for row in range(4):
            if row != state[col]:
                new_state = state.copy()
                new_state[col] = row
                neighbors.append(new_state)
    return neighbors


# Hill Climbing with Side-Steps to escape local minima
def hill_climbing(start, max_sidesteps=100):
    current = start
    current_h = heuristic(current)
    sidesteps = 0

    while True:
        neighbors = get_neighbors(current)

        # Find the best neighbors (minimize conflicts)
        best_neighbors = []
        best_h = current_h

        for neighbor in neighbors:
            h = heuristic(neighbor)
            if h < best_h:
                best_neighbors = [neighbor]
                best_h = h
            elif h == best_h:
                best_neighbors.append(neighbor)

        # If no neighbor is better or equal
        if best_h > current_h:
            break

        # If the best found is strictly better
        if best_h < current_h:
            current = random.choice(best_neighbors)
            current_h = best_h
            sidesteps = 0
        # If the best found is equal (side-step)
        elif best_h == current_h:
            if sidesteps < max_sidesteps and best_neighbors:
                current = random.choice(best_neighbors)
                current_h = best_h
                sidesteps += 1
            else:
                break

    return current


# Print board
def print_board(state):
    for row in range(4):
        for col in range(4):
            if state[col] == row:
                print("Q", end=" ")
            else:
                print(".", end=" ")
        print()


# -----------------------------
# MAIN RUNNER
# -----------------------------

print("4 Queens - Hill Climbing")

print("\nEnter initial state.")
print("Enter 4 numbers (0-3).")
print("Each number represents the row of the queen in each column.")
print("\nExample: 1 3 0 2")

try:
    user_input = input("\nInitial state: ").strip()
    # Fallback default if input is empty
    if not user_input:
        user_input = "0 2 3 1"
        print(f"Using default state: {user_input}")
    start = list(map(int, user_input.split()))
except Exception as e:
    start = [0, 2, 3, 1]
    print(f"Invalid input, using default state: 0 2 3 1")

# Check input
if len(start) != 4 or any(x < 0 or x > 3 for x in start):
    print("Invalid input!")
    print("Enter 4 numbers between 0 and 3.")
else:
    print("\nInitial State:")
    print_board(start)
    print("\nInitial heuristic (conflicts):", heuristic(start))

    # Apply Hill Climbing
    solution = hill_climbing(start)

    print("\nFinal State:")
    print_board(solution)
    print("\nFinal heuristic (conflicts):", heuristic(solution))

    if heuristic(solution) == 0:
        print("\nSolution found!")
    else:
        print("\nHill Climbing got stuck at a local minimum.")

4 Queens - Hill Climbing

Enter initial state.
Enter 4 numbers (0-3).
Each number represents the row of the queen in each column.

Example: 1 3 0 2

Initial state: 0 2 3 1

Initial State:
Q . . . 
. . . Q 
. Q . . 
. . Q . 

Initial heuristic (conflicts): 1

Final State:
. Q . . 
. . . Q 
Q . . . 
. . Q . 

Final heuristic (conflicts): 0

Solution found!
